# Finish: merge the trained adapter + export GGUF

Use this when training already ran and the LoRA adapter was checkpointed, but the
merge/convert step didn't finish. It skips retraining: attach the training run's
output as an **Input** (right sidebar) so `receipt-extractor-ckpt` is available,
then Run All. Needs **GPU T4 x2** and **Internet On**.

## 1. Dependencies

In [ ]:
!pip -q install "transformers>=4.44" "peft>=0.12" "accelerate>=0.33" "bitsandbytes>=0.43"

# Kaggle preinstalls an old torchao that current peft rejects during the merge.
!pip -q uninstall -y torchao

## 2. Locate the checkpointed adapter

The training run saved it to `/kaggle/working/receipt-extractor-ckpt`. Once you add
that run's output as an Input, it appears under `/kaggle/input/...`.

In [ ]:
import glob, os
hits = glob.glob('/kaggle/input/**/adapter_config.json', recursive=True)
assert hits, 'No adapter found - add the training run output as an Input (right sidebar).'
ADAPTER_DIR = os.path.dirname(hits[0])
print('using adapter at', ADAPTER_DIR)

## 3. Merge on the GPU

CPU RAM (~16 GB) can't hold an fp16 7B; the freed T4 x2 VRAM (~30 GB) can. `merge_and_unload` folds the LoRA deltas into the base weights.

In [ ]:
import gc, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE_MODEL = 'Qwen/Qwen2.5-7B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

base_fp16 = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.float16, device_map='auto')
merged = PeftModel.from_pretrained(base_fp16, ADAPTER_DIR).merge_and_unload()

# /tmp, not the working dir: merged (~15 GB) + gguf intermediate (~15 GB) would
# overflow the ~19.5 GB /kaggle/working quota.
merged.save_pretrained('/tmp/merged', safe_serialization=True)
tokenizer.save_pretrained('/tmp/merged')

del base_fp16, merged
gc.collect(); torch.cuda.empty_cache()
print('merged -> /tmp/merged')

## 4. Convert to quantized GGUF

Only the final ~4.5 GB Q4 lands in the working dir, for download.

In [ ]:
import os, shutil
!git clone --depth 1 https://github.com/ggerganov/llama.cpp
!pip -q install -r llama.cpp/requirements.txt

F16 = '/tmp/f16.gguf'
Q4  = '/kaggle/working/receipt-extractor-q4.gguf'
!python llama.cpp/convert_hf_to_gguf.py /tmp/merged --outfile {F16} --outtype f16
shutil.rmtree('/tmp/merged', ignore_errors=True)
!cd llama.cpp && cmake -B build && cmake --build build --config Release -j
!./llama.cpp/build/bin/llama-quantize {F16} {Q4} Q4_K_M
os.remove(F16)
print('GGUF ready at', Q4, '- download it from the Output panel.')